In [1]:
Lab Session: Exploring Intelligent Agents & Developing Agent Programs
# Course:Artificial Intelligence  
# Topic:Intelligent Agents  
# Date: October 8, 2025
🎯 Objectives:
# - Understand the concept of Intelligent Agents.
# - Implement Reflex and Model-Based Agents.
# - Apply agent programs to real-world-like problems.
# - Practice reasoning and decision-making logic in AI systems

# Define a Basic Agent... Vaccum clearner agent 

In [7]:
# Define a basic Agent class
class Agent:
    def __init__(self):
        # Initialize the agent's percept to None
        self.percept = None

    def perceive(self, environment):
        # Agent receives a percept from the environment using its sense() method
        self.percept = environment.sense()
        # Display what the agent perceives
        print(f"Agent perceives: {self.percept}")

    def act(self):
        # Agent takes an action (not based on any logic yet)
        print("Agent takes an action based on the percept.")

# Define a simple environment that the agent can interact with
class SimpleEnvironment:
    def sense(self):
        # The environment always returns a fixed percept (e.g., detecting dirt)
        return "Dirt Detected"

# Create an instance of the environment
env = SimpleEnvironment()

# Create an instance of the agent
agent = Agent()

# Agent receives percept from the environment
agent.perceive(env)

# Agent performs an action based on the percept
agent.act()


Agent perceives: Dirt Detected
Agent takes an action based on the percept.


# simple reflex action 

In [ ]:

class ReflexVacuumAgent:
    def decide(self, percept):
        # Decision is made based on simple condition: if dirt, clean it; else move
        if percept == "Dirt":
            return "Suck"
        else:
            return "Move Right"

    def act(self, environment):
        # Get the percept from the environment
        percept = environment.sense()
        # Decide action based on the percept
        action = self.decide(percept)
        # Display the percept and action
        print(f"Percept: {percept} => Action: {action}")

# Environment that can either be "Dirt" or "Clean"
class VacuumEnvironment:
    def __init__(self, state):
        # Save the current state of the environment
        self.state = state

    def sense(self):
        # Return the current state as the percept
        return self.state

# Instantiate environment with "Dirt" state
env = VacuumEnvironment("Dirt")

# Create the reflex agent
agent = ReflexVacuumAgent()

# Run the agent in the environment
agent.act(env)


: 

# Weather-Based Decision Agent
#agent immediately decides an action based only on the current weather percept.

In [5]:

import random

class WeatherEnvironment:
    def sense(self):
        # Randomly return a weather condition
        return random.choice(["Sunny", "Rainy", "Cloudy"])

class WeatherAgent:
    def decide(self, weather):
        # Make a decision based on the weather condition
        if weather == "rainy":
            return "Go Outside"
        elif weather == "Rainy":
            return "Stay Inside"
        else:
            return "Maybe take an umbrella"

    def act(self, environment):
        # Get current weather
        weather = environment.sense()
        # Decide what to do based on the weather
        action = self.decide(weather)
        # Print the decision
        print(f"Weather: {weather} => Action: {action}")

# Create environment and agent
env = WeatherEnvironment()
agent = WeatherAgent()

# Run the agent multiple times to see different outcomes
for _ in range(1):
    agent.act(env)


Weather: Rainy => Action: Stay Inside


# using model based weather agents

In [3]:
import random

class WeatherEnvironment:
    def sense(self):
        # Randomly return a weather condition
        return random.choice(["Sunny", "Rainy", "Cloudy"])

class ModelBasedWeatherAgent:
    def __init__(self):
        # Internal memory to store weather history
        self.history = []

    def update_state(self, weather):
        # Remember the current weather
        self.history.append(weather)

    def decide(self, weather):
        # Update internal state first
        self.update_state(weather)
        
        # Decision based on current and past weather
        if weather == "Sunny":
            return "Go Outside"
   
        elif weather == "Rainy":
            # If it rained yesterday too, suggest staying home longer
            if len(self.history) >= 2 and self.history[-2] == "Rainy":
                return "Stay Inside and take precautions"
            return "Stay Inside"
        else:  # Cloudy
            # If cloudy after sunny days, suggest carrying an umbrella just in case
            if len(self.history) >= 2 and self.history[-2] == "Sunny":
                return "Maybe take an umbrella"
            return "Check umbrella"

    def act(self, environment):
        # Get current weather
        weather = environment.sense()
        # Decide action based on current weather and memory
        action = self.decide(weather)
        # Print percept, memory, and action
        print(f"Weather: {weather}, History: {self.history}, Action: {action}")

# Create environment and model-based agent
env = WeatherEnvironment()
agent = ModelBasedWeatherAgent()

# Run agent multiple times to see how it reacts using memory
for _ in range(5):
    agent.act(env)


Weather: Rainy, History: ['Rainy'], Action: Stay Inside
Weather: Cloudy, History: ['Rainy', 'Cloudy'], Action: Check umbrella
Weather: Sunny, History: ['Rainy', 'Cloudy', 'Sunny'], Action: Go Outside
Weather: Rainy, History: ['Rainy', 'Cloudy', 'Sunny', 'Rainy'], Action: Stay Inside
Weather: Cloudy, History: ['Rainy', 'Cloudy', 'Sunny', 'Rainy', 'Cloudy'], Action: Check umbrella


# Real Weather + Model-Based Agent for Debre Birhan using API from Open weather

In [7]:
import requests

# ---------------- Environment ----------------
class WeatherEnvironment:
    def __init__(self, lat, lon, api_key):
        self.lat = lat
        self.lon = lon
        self.api_key = api_key

    def sense(self):
        # Correct OpenWeatherMap weather API URL
        url = "https://api.openweathermap.org/data/2.5/weather"

        # Send coordinates and API key to the weather API
        params = {
            "lat": self.lat,
            "lon": self.lon,
            "appid": self.api_key,
            "units": "metric"
        }

        try:
            response = requests.get(url, params=params)

            # Check whether the request was successful
            if response.status_code == 200:

                data = response.json()

                # Extract weather condition and temperature
                weather = data["weather"][0]["main"]
                temp = data["main"]["temp"]

                print(
                    f"Weather at coordinates "
                    f"({self.lat}, {self.lon}): "
                    f"{weather}, Temperature: {temp}°C"
                )

                return weather

            else:
                # Handle API errors
                try:
                    data = response.json()
                    message = data.get("message", "Unknown error")
                except ValueError:
                    message = response.text

                print(
                    f"Error fetching weather "
                    f"(Status Code {response.status_code}): {message}"
                )

                return None

        except requests.exceptions.RequestException as e:
            print("Connection error:", e)
            return None


# ---------------- Model-Based Agent ----------------
class ModelBasedWeatherAgent:
    def __init__(self):
        # Internal memory to store weather history
        self.history = []

    def update_state(self, weather):
        # Remember the current weather
        self.history.append(weather)

    def decide(self, weather):
        # Update internal state
        self.update_state(weather)

        # Decision based on current weather and history
        if weather == "Clear":
            return "Go Outside"

        elif weather == "Rain":

            # Check if it also rained previously
            if len(self.history) >= 2 and self.history[-2] == "Rain":
                return "Stay Inside and take precautions"

            return "Stay Inside"

        else:
            # Clouds or other weather conditions
            return "Maybe take an umbrella"

    def act(self, environment):

        # Get current weather from the environment
        weather = environment.sense()

        # Make a decision if weather was successfully received
        if weather:

            action = self.decide(weather)

            print(f"Agent History: {self.history}")
            print(f"Action: {action}")

        print("-" * 50)


# ---------------- Usage ----------------
api_key = "154241d175aa2a4080374357a50398ad"

# Debre Birhan coordinates
lat = 9.6796
lon = 39.5333

# Create the environment
env = WeatherEnvironment(lat, lon, api_key)

# Create the model-based agent
agent = ModelBasedWeatherAgent()

# Run the agent 3 times
for _ in range(3):
    agent.act(env)

Weather at coordinates (9.6796, 39.5333): Clouds, Temperature: 19.6°C
Agent History: ['Clouds']
Action: Maybe take an umbrella
--------------------------------------------------
Weather at coordinates (9.6796, 39.5333): Clouds, Temperature: 19.6°C
Agent History: ['Clouds', 'Clouds']
Action: Maybe take an umbrella
--------------------------------------------------
Weather at coordinates (9.6796, 39.5333): Clouds, Temperature: 19.6°C
Agent History: ['Clouds', 'Clouds', 'Clouds']
Action: Maybe take an umbrella
--------------------------------------------------


# class exercise

In [9]:
# 1. Classroom Temperature Monitor
# Scenario

# - An agent monitors classroom temperature and decides whether to open windows or turn on AC.

# Percepts: "Temperature" (numerical) and "Occupied" (True/False)

# Rules

# If "Occupied" and "Temperature > 25" → "Turn on AC"

# If "Occupied" and "Temperature < 18" → "Open Windows"

# If "Empty" → "Do Nothing"
# what u do?
# Implement a Simple Reflex Agent using these rules.
# Then extend to Model-Based by remembering last action and temperature trends.

def simple_reflex_agent(temperature, occupied):
    if occupied:
        if temperature > 25:
            return "Turn on AC"
        elif temperature < 18:
            return "Open Windows"
        else:
            return "Do Nothing"
    else:
        return "Do Nothing"
print(simple_reflex_agent(temperature=27, occupied=True))  # Example usage

Do Nothing


In [ ]:
# Simple Reflex Mail Sorting Agent
class SimpleReflexMailSortingAgent:
    def classify(self, tag):
        tag_text = str(tag).lower()
        if "boss" in tag_text:
            return "Work"
        elif "friend" in tag_text:
            return "Personal"
        elif "advertisement" in tag_text or "ad" in tag_text:
            return "Spam"
        else:
            return "Personal"  # default fallback

    def sort(self, tag):
        return self.classify(tag)


# Model-Based Mail Sorting Agent
class ModelBasedMailSortingAgent:
    def __init__(self, storage=None, storage_bins=None, bin_capacity=5):
        self.storage = storage if storage is not None else {"Work": 0, "Personal": 0, "Spam": 0}
        self.storage_bins = storage_bins if storage_bins is not None else ["Work", "Personal", "Spam"]
        self.bin_capacity = bin_capacity
        self.last_action = None
        self.last_tag = None

    def classify(self, tag):
        tag_text = str(tag).lower()
        if "boss" in tag_text:
            return "Work"
        elif "friend" in tag_text:
            return "Personal"
        elif "advertisement" in tag_text or "ad" in tag_text:
            return "Spam"
        else:
            return "Personal"

    def sort(self, tag):
        category = self.classify(tag)
        self.storage[category] = self.storage.get(category, 0) + 1
        self.last_tag = tag
        self.last_action = f"Sorted '{tag}' into {category}"
        
        if self.storage[category] >= self.bin_capacity:
            print(f"Alert: {category} bin is full ({self.storage[category]}/{self.bin_capacity})")
        
        return category


# Reuse notebook variables if they already exist
if "mail_agent" not in globals():
    mail_agent = ModelBasedMailSortingAgent(
        storage=globals().get("storage", {"Work": 0, "Personal": 0, "Spam": 0}),
        storage_bins=globals().get("storage_bins", ["Work", "Personal", "Spam"]),
        bin_capacity=5
    )

# Example usage
for letter in ["From Boss", "From Friend", "Advertisement", "From Boss", "From Friend"]:
    result = mail_agent.sort(letter)
    print(f"{letter} -> {result}")

print("Current storage:", mail_agent.storage)


TypeError: ModelBasedMailSortingAgent.__init__() missing 1 required positional argument: 'rules'

In [ ]:
Smart Classroom Lighting
An agent monitors the classroom light level and whether the classroom is occupied.
The agent decides whether to turn the lights on or off.

Percepts: "Light Level" (Dark/Bright) and "Occupied" (True/False)

Rules
If "Occupied" and "Dark" → "Turn on lights"
If "Occupied" and "Bright" → "Turn on lights"
If "Empty" → "Turn off lights"
What do you do?

1. Implement a Simple Reflex Agent using the corrected rules.

2. Extend the agent to a Model-Based Agent by remembering:

the last action
the previous light level
the previous occupancy state

In [ ]:
Scenario: Smart Refrigerator

An agent monitors the refrigerator and decides whether to turn the cooling system on or off.

Percepts:

"Temperature" (numerical)
"Door Open" (True/False)
"Food Present" (True/False)
Rules
If "Temperature > 5" and "Food Present" → "Turn ON cooling"
If "Temperature ≤ 5" and "Food Present" → "Turn OFF cooling"
If "Door Open" → "Turn OFF cooling"
If "Food Present" = False → "Turn ON cooling"
What do you do?

1. Implement a Simple Reflex Agent using the rules.

2. Extend it to a Model-Based Agent by remembering:

the previous temperature
the last action
how long the door has been open

In [ ]:
Scenario: Smart Hospital Room Environment Agent

An intelligent agent monitors a hospital patient's room and decides whether to turn on the AC, open the window, or do nothing.

Percepts
"Temperature" — numerical value in °C
"Patient Present" — True/False
"Window Open" — True/False
"Air Quality" — Good/Poor
"Emergency" — True/False
Rules
If "Patient Present" and "Temperature > 26" → "Turn ON AC"
If "Patient Present" and "Temperature < 18" → "Open Window"
If "Patient Present" and "Air Quality = Poor" → "Open Window"
If "Window Open" and "Temperature > 26" → "Close Window"
If "Emergency" → "Do Nothing"
If "Patient Present" = False → "Turn OFF AC"
What do you do?

1. Implement a Simple Reflex Agent using the corrected rules.

2. Then extend the agent to a Model-Based Agent by remembering:
Previous temperature
Previous air quality
Previous action
Previous window state
How long the patient has been in the room